# CAFA5 Minimal Graph Training

Objective: run a small, reproducible graph-training smoke test before larger experiments.

Success criteria:
- Use the existing graph cache and graph split manifests.
- Create a small, non-overwriting smoke split for one GO aspect.
- Run `train_minimal_graph_model.py` end-to-end with PyG.
- Save and summarize a checkpoint run under `graph_cache/training_runs/`.

In [ ]:
# Setup and configuration
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> tuple[Path, str]:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate, 'cwd_search'
    return start, 'cwd_fallback'

def resolve_repo_root() -> tuple[Path, str]:
    configured = os.environ.get('CAFA5_REPO_ROOT')
    if configured:
        return Path(configured).expanduser().resolve(), 'env:CAFA5_REPO_ROOT'
    return find_repo_root(Path.cwd())

REPO_ROOT, REPO_ROOT_SOURCE = resolve_repo_root()
SERVER_USER = os.environ.get('USER', 'bensonli')
RUN_ROOT = Path(os.environ.get('CAFA5_RUN_ROOT', f'/global/scratch/users/{SERVER_USER}/cafa5_outputs'))
GRAPH_CACHE_DIR = RUN_ROOT / 'graph_cache'
GRAPH_SPLIT_DIR = GRAPH_CACHE_DIR / 'splits'
GRAPH_PYTHON = Path(os.environ.get('CAFA5_GRAPH_PYTHON', sys.executable))

FRAMEWORK = os.environ.get('CAFA5_TRAIN_FRAMEWORK', 'pyg')
ASPECT = os.environ.get('CAFA5_TRAIN_ASPECT', 'MFO').upper()
MIN_TERM_FREQUENCY = int(os.environ.get('CAFA5_MIN_TERM_FREQUENCY', '20'))
DEVICE = os.environ.get('CAFA5_TRAIN_DEVICE', 'auto')

TRAIN_LIMIT = int(os.environ.get('CAFA5_SMOKE_TRAIN_LIMIT', '256'))
VAL_LIMIT = int(os.environ.get('CAFA5_SMOKE_VAL_LIMIT', '64'))
TEST_LIMIT = int(os.environ.get('CAFA5_SMOKE_TEST_LIMIT', '64'))
EPOCHS = int(os.environ.get('CAFA5_SMOKE_EPOCHS', '2'))
BATCH_SIZE = int(os.environ.get('CAFA5_SMOKE_BATCH_SIZE', '8'))
HIDDEN_DIM = int(os.environ.get('CAFA5_SMOKE_HIDDEN_DIM', '64'))
NUM_WORKERS = int(os.environ.get('CAFA5_SMOKE_NUM_WORKERS', '0'))

RUN_TRAINING = os.environ.get('CAFA5_RUN_TRAINING', '1') == '1'
OVERWRITE_SMOKE_SPLITS = os.environ.get('CAFA5_OVERWRITE_SMOKE_SPLITS', '0') == '1'

SMOKE_SPLIT_DIR = GRAPH_CACHE_DIR / f'splits_smoke_{ASPECT.lower()}_t{TRAIN_LIMIT}_v{VAL_LIMIT}_s{TEST_LIMIT}'
CHECKPOINT_DIR = GRAPH_CACHE_DIR / 'training_runs' / f'minimal_{FRAMEWORK}_{ASPECT.lower()}_smoke'

config = {
    'repo_root': str(REPO_ROOT),
    'repo_root_source': REPO_ROOT_SOURCE,
    'run_root': str(RUN_ROOT),
    'graph_python': str(GRAPH_PYTHON),
    'graph_cache_dir': str(GRAPH_CACHE_DIR),
    'source_split_dir': str(GRAPH_SPLIT_DIR),
    'smoke_split_dir': str(SMOKE_SPLIT_DIR),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'framework': FRAMEWORK,
    'aspect': ASPECT,
    'min_term_frequency': MIN_TERM_FREQUENCY,
    'device': DEVICE,
    'limits': {'train': TRAIN_LIMIT, 'val': VAL_LIMIT, 'test': TEST_LIMIT},
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'hidden_dim': HIDDEN_DIM,
    'num_workers': NUM_WORKERS,
    'run_training': RUN_TRAINING,
    'overwrite_smoke_splits': OVERWRITE_SMOKE_SPLITS,
}
print(json.dumps(config, indent=2))

required_repo_files = [REPO_ROOT / 'train_minimal_graph_model.py', REPO_ROOT / 'cafa_graph_dataloaders.py']
missing_repo_files = [str(path) for path in required_repo_files if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError(f'REPO_ROOT is wrong or incomplete: {missing_repo_files}')

## Plan

This notebook follows the planned smoke-test path: verify the graph Python environment, derive a small split from the full graph splits, train a minimal PyG classifier, then inspect the summary JSON. Keep these defaults small; increase the limits only after the run is stable.

In [ ]:
# Helper for live command output
def format_cmd(cmd: list[object]) -> str:
    return ' '.join(shlex.quote(str(part)) for part in cmd)

def run_live(cmd: list[object], cwd: Path | None = None) -> None:
    cmd_str = format_cmd(cmd)
    print(f'[run] {cmd_str}', flush=True)
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd or REPO_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, [str(part) for part in cmd])

In [ ]:
# Verify the graph training environment before touching data.
env_check = [
    GRAPH_PYTHON,
    '-c',
    (
        'import sys; import torch; import torch_geometric; '
        'print("python", sys.executable); '
        'print("torch", torch.__version__); '
        'print("torch_geometric", torch_geometric.__version__); '
        'print("cuda_available", torch.cuda.is_available())'
    ),
]
run_live(env_check)

In [ ]:
# Create a small smoke split from the existing graph split manifests.
source_aspect_dir = GRAPH_SPLIT_DIR / ASPECT.lower()
smoke_aspect_dir = SMOKE_SPLIT_DIR / ASPECT.lower()
split_limits = {'train': TRAIN_LIMIT, 'val': VAL_LIMIT, 'test': TEST_LIMIT}

if not source_aspect_dir.exists():
    raise FileNotFoundError(
        f'Missing source split directory: {source_aspect_dir}. '
        'Run export_graph_dataloaders.py first.'
    )

def read_entry_ids(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def write_lines_if_needed(path: Path, lines: list[str], overwrite: bool = False) -> None:
    if path.exists() and not overwrite:
        print(f'[skip] already exists: {path}')
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{line}\n' for line in lines), encoding='utf-8')
    print(f'[write] {path} ({len(lines)} lines)')

smoke_summary = {
    'source_split_dir': str(source_aspect_dir),
    'root': str(SMOKE_SPLIT_DIR),
    'aspect': ASPECT,
    'min_term_frequency': MIN_TERM_FREQUENCY,
    'limits': split_limits,
    'counts': {},
    'source_counts': {},
}

for split_name, limit in split_limits.items():
    src = source_aspect_dir / f'{split_name}.txt'
    if not src.exists():
        raise FileNotFoundError(f'Missing source split file: {src}')
    ids = read_entry_ids(src)
    subset = ids[:min(limit, len(ids))]
    smoke_summary['source_counts'][split_name] = len(ids)
    smoke_summary['counts'][split_name] = len(subset)
    write_lines_if_needed(
        smoke_aspect_dir / f'{split_name}.txt',
        subset,
        overwrite=OVERWRITE_SMOKE_SPLITS,
    )

source_summary_path = source_aspect_dir / 'summary.json'
if source_summary_path.exists():
    write_lines_if_needed(
        smoke_aspect_dir / 'summary.json',
        source_summary_path.read_text(encoding='utf-8').splitlines(),
        overwrite=OVERWRITE_SMOKE_SPLITS,
    )

smoke_summary_path = SMOKE_SPLIT_DIR / 'summary.json'
if smoke_summary_path.exists() and not OVERWRITE_SMOKE_SPLITS:
    print(f'[skip] already exists: {smoke_summary_path}')
else:
    smoke_summary_path.parent.mkdir(parents=True, exist_ok=True)
    smoke_summary_path.write_text(json.dumps(smoke_summary, indent=2), encoding='utf-8')
    print(f'[write] {smoke_summary_path}')

display(pd.DataFrame([
    {'split': split_name, 'selected': smoke_summary['counts'][split_name], 'source_total': smoke_summary['source_counts'][split_name]}
    for split_name in ('train', 'val', 'test')
]))

In [ ]:
# Run the minimal graph training script on the smoke split.
train_cmd = [
    GRAPH_PYTHON,
    REPO_ROOT / 'train_minimal_graph_model.py',
    '--root', GRAPH_CACHE_DIR,
    '--split-dir', SMOKE_SPLIT_DIR,
    '--framework', FRAMEWORK,
    '--aspect', ASPECT,
    '--epochs', EPOCHS,
    '--batch-size', BATCH_SIZE,
    '--num-workers', NUM_WORKERS,
    '--hidden-dim', HIDDEN_DIM,
    '--device', DEVICE,
    '--min-term-frequency', MIN_TERM_FREQUENCY,
    '--checkpoint-dir', CHECKPOINT_DIR,
]

if RUN_TRAINING:
    run_live(train_cmd)
else:
    print('[skip] RUN_TRAINING is False')
    print(format_cmd(train_cmd))

In [ ]:
# Summarize the smoke training run.
summary_path = CHECKPOINT_DIR / 'summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Training summary is missing: {summary_path}')

training_summary = json.loads(summary_path.read_text(encoding='utf-8'))
rows = []
for record in training_summary.get('history', []):
    row = {
        'epoch': record['epoch'],
        'epoch_seconds': record['epoch_seconds'],
    }
    for split_name in ('train', 'val', 'test'):
        metrics = record.get(split_name, {})
        row[f'{split_name}_loss'] = metrics.get('loss')
        row[f'{split_name}_micro_f1'] = metrics.get('micro_f1')
        row[f'{split_name}_graphs'] = metrics.get('graphs')
    rows.append(row)

metrics_df = pd.DataFrame(rows)
display(metrics_df)

key_summary = {
    'framework': training_summary.get('framework'),
    'aspect': training_summary.get('aspect'),
    'device': training_summary.get('device'),
    'epochs': training_summary.get('epochs'),
    'batch_size': training_summary.get('batch_size'),
    'hidden_dim': training_summary.get('hidden_dim'),
    'best_val_loss': training_summary.get('best_val_loss'),
    'best_checkpoint_path': training_summary.get('best_checkpoint_path'),
}
print(json.dumps(key_summary, indent=2))

## Notes and Next Steps

Record the observed train/val/test loss and micro-F1 here after the run. If this smoke run finishes cleanly, the next step is to increase `TRAIN_LIMIT`, `VAL_LIMIT`, and `TEST_LIMIT`, then compare aspects (`BPO`, `CCO`, `MFO`) or move from `DEVICE=auto` to a GPU-specific run.